In [21]:
import reframed
import numpy as np
import pandas as pd
from pathlib import Path
import sys
import logging
import time

sys.path.append('../../code/7_GEM_reconstruction')
from fix_universe import *
from ng_utils import fix_compartments, fix_reframed_annotations, check_reaction_balance
from model_qc_and_polish import curate_model, fix_biomass

# This notebook is used to make small modifications to the carveme bacteria universe before reconstruction

In [2]:
repo_folder = Path("../..")
data_folder = repo_folder / "data" / "7_GEM_reconstruction"
growth_data_folder = repo_folder / 'data' / '1_growth_phenotyping'

In [3]:
log_folder = repo_folder / 'notebooks' / '7_GEM_reconstruction' / 'log'
timestr = time.strftime("%Y%m%d_%H%M")
logfile = log_folder / f'fix_universe_{timestr}.log'
logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s %(levelname)s:%(name)-8s %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S',
                     handlers=[logging.FileHandler(str(logfile)),logging.StreamHandler()])
logging.captureWarnings(True)

In [4]:
modelu = reframed.load_cbmodel(growth_data_folder / "carveme_universe_bacteria.xml")

In [5]:
logger = logging.getLogger("fix_universe")

In [6]:
logger.info('Fix compartment abbreviations')
fix_compartments(modelu)

2026-01-07 18:19:01 INFO:fix_universe Fix compartment abbreviations


In [7]:
add_biotin_synthase_reactions(modelu, logger = logger)

2026-01-07 18:19:03 INFO:fix_universe Added biotin synthase reactions ['R_MALCOAMT', 'R_OGMEACPS', 'R_OGMEACPR', 'R_OPMEACPS', 'R_OPMEACPD', 'R_EPMEACPR', 'R_AOXSr2', 'R_OGMEACPD', 'R_EGMEACPR', 'R_OPMEACPR', 'R_PMEACPE', 'R_BTS5']


In [8]:
fix_misc(modelu, logger = logger)

2026-01-07 18:19:03 INFO:fix_universe Fixing miscellaneous issues in the universe model


In [9]:
remove_metabolites_without_exchange_reactions(modelu, logger)

In [10]:
curate_model(modelu, logger)

2026-01-07 18:19:03 INFO:fix_universe Deleted these metabolites because they were duplicates: M_1btol_c, M_2h3oppan_c, M_2hymeph_c, M_34dhpha_c, M_3hbycoa_c, M_5dh4dglc_c, M_amacald_c, M_abt_c, M_abt_e, M_nal2a6o_c, M_trans_dd2coa_c, M_ethso3_c, M_ethso3_e, M_galctr__D_c, M_galctr__D_e, M_glcn_c, M_glcn_e, M_istnt_c, M_istnt_e, M_metsox_S__L_c, M_metsox_S__L_e, M_orn__L_c, M_orn__L_e, M_sula_c, M_sula_e
2026-01-07 18:19:06 INFO:fix_universe Deleted these reactions because they are duplicates: R_ASPA2, R_CMLDC, R_4HTHRS_1, R_HCITS, R_ACGK_1, R_ACODA_1, R_ACS_1, R_ACYP_3, R_ADK2_1, R_AKP1_2, R_ALKP_1, R_ATAH_1, R_AMANK_1, R_AMID_1, R_TMN, R_PTPATi, R_ARGDr, R_ASPCT_2, R_CYSTA, R_BPNT_1, R_BTS_1, R_UPPN, R_CO23OC, R_CBMKr2, R_CBPS_1, R_DTTGY, R_CDPMEK_1, R_CHORS_1, R_TRPAS1, R_CYSS_2, R_CYTDK1_1, R_CYTDK2_1, R_DGTCY, R_DABAAT2, R_DATCY, R_DADNK_1, R_DCTCP, R_DDPA_1, R_DGNSK_1, R_DHBSH_1, R_DHNPA_1, R_DHPM1, R_DHQS_2, R_METRR, R_DPCOAK_1, R_DRBK_1, R_DUTCP, R_DURIK1_1, R_DURIPP_1, R_FCLT_2

In [11]:
fix_biomass(modelu, logger = logger, add_biotin=False)

2026-01-07 18:19:06 INFO:fix_universe Biomass equationsums to 1.0327153669083737 -> rescaling
2026-01-07 18:19:06 INFO:fix_universe Rescaled biomass equation from 1.0327153669083737 to 1.000000756197047


# Save

In [12]:
reframed.save_cbmodel(modelu, str(data_folder / "carveme_universe_bacteria_fixed.xml"))

In [16]:
r = modelu.reactions.R_KGD2

In [19]:
r.stoichiometry['M_h_c'] = -2

In [24]:
check_reaction_balance(modelu, r)

(False,
 0.001007939999999874,
 False,
 defaultdict(int,
             {'C': 0.0,
              'O': 0.0,
              'Fe': 0.0,
              'S': 0.0,
              'H': -1.0,
              'N': 0.0,
              'P': 0.0}),
 False,
 -1.0)